# 30 · SARIMAX por UF com 16 exógenas simultâneas

Cada UF/período contém uma única matriz X de 16 tipologias. Quando há constantes e posto deficiente, a especificação completa não é identificável; não substituímos por uma regressão reduzida.

## 1. Entrada, integridade e objetivos

Os códigos desta revisão não substituem os arquivos originais; os dados consolidados devem ser fornecidos localmente. A inferência é exploratória.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
SAIDA=ROOT/'outputs/tables/revisao_estadual_20261019'
assert SAIDA.exists()
print('Repositório:',ROOT)
print('Base:',os.getenv('TCC_BASE_CSV',str(ROOT/'data/processed/df_tcc_2013_2024.csv')))


Repositório: /mnt/data/tcc_revisao/repo
Base: /mnt/data/tcc_bases/df_tcc_2013_2024_recovered.csv


## 2. Especificação e identificabilidade

`y_it=c+Σ_{j=1}^{16}β_j Δlog(1+D_{j,it})+u_it`, com erro SARIMA selecionado no treino. Todas as 16 exposições contemporâneas devem entrar simultaneamente. Teste LR conjunto H₀: β₁=…=β₁₆=0 quando os ajustes aninhados convergem e têm mesma amostra. Testes individuais BH entre as 16 por UF×período.

In [2]:
models=pd.concat([pd.read_csv(SAIDA/f"sarimax_modelos_{p}.csv") for p in ["Pré","Total"]]);display(models[["periodo","uf","rank_total","rank_treino","status"]])

,periodo,uf,rank_total,rank_treino,status
0,Pré,AC,9,6,16 coeficientes não identificáveis no período:...
1,Pré,AL,7,7,16 coeficientes não identificáveis no período:...
2,Pré,AM,11,11,16 coeficientes não identificáveis no período:...
3,Pré,AP,8,8,16 coeficientes não identificáveis no período:...
4,Pré,BA,13,12,16 coeficientes não identificáveis no período:...
5,Pré,CE,10,10,16 coeficientes não identificáveis no período:...
6,Pré,DF,3,2,16 coeficientes não identificáveis no período:...
7,Pré,ES,15,14,16 coeficientes não identificáveis no período:...
8,Pré,GO,11,11,16 coeficientes não identificáveis no período:...
9,Pré,MA,7,7,16 coeficientes não identificáveis no período:...


## 3. Comparação em estados com posto completo

In [3]:
adequados=models[models.rank_total==16];display(adequados[["uf","periodo","p_conjunto","delta_BIC","RMSE_SARIMA","RMSE_SARIMAX","variacao_RMSE_pct","status"]])

,uf,periodo,p_conjunto,delta_BIC,RMSE_SARIMA,RMSE_SARIMAX,variacao_RMSE_pct,status
10,MG,Total,0.349290,61.829300,1.356279,1.286058,-5.177514,16 exógenas estimadas; comparação preditiva co...
11,MS,Total,0.284340,60.702046,4.496684,NaN,NaN,16 exógenas estimadas somente no período compl...
22,RS,Total,0.236450,59.756569,1.731246,NaN,NaN,16 exógenas estimadas somente no período compl...
23,SC,Total,0.816588,68.521508,1.504725,NaN,NaN,16 exógenas estimadas somente no período compl...
25,SP,Total,0.494894,63.996278,1.360490,NaN,NaN,16 exógenas estimadas somente no período compl...


## 4. Hipóteses individuais

In [4]:
co=pd.read_csv(SAIDA/"sarimax_coeficientes_Total.csv");print("Nominais:",co.rejeicao_nominal.sum(),"BH:",co.rejeicao_bh.sum());display(co[co.rejeicao_nominal][["uf","tipologia","coeficiente","p","q_bh"]])

Nominais: 2 BH: 0


,uf,tipologia,coeficiente,p,q_bh
29,MS,tipo_rompimento_colapso_de_barragens,6.448356,0.010842,0.173467
37,RS,tipo_estiagem_e_seca,0.529674,0.020001,0.320018


## 5. Conclusão

Nenhum teste conjunto da regressão de 16 exógenas rejeita a 5%. Todas as cinco UFs com posto completo têm BIC SARIMAX superior ao SARIMA. Apenas MG possui treino com posto 16 e comparação de RMSE fora da amostra; a previsão usa o desastre observado no próprio mês.